# Run both corrected N=16 notebooks overnight

Run this notebook once. It executes the phase-plus-descent notebook first and the corrected log-weight-variance notebook second. Both save incremental CSV results, checkpoints and loss curves beneath one timestamped `runs/overnight_...` directory.


In [ ]:
import os
import traceback
from datetime import datetime
from pathlib import Path

ip = get_ipython()
names = [
    "Complex_Time_QHO_Anharmonic(3).ipynb",
    "Complex_Time_QHO_Anharmonic_MU(1).ipynb",
]

search_roots = [Path.cwd(), Path.cwd() / "upload"]
resolved = []
for name in names:
    matches = [root / name for root in search_roots if (root / name).is_file()]
    if not matches:
        raise FileNotFoundError(
            f"Could not find {name}. Put this launcher in the same folder as both notebooks.")
    resolved.append(matches[0].resolve())

overnight_root = Path.cwd() / "runs" / f"overnight_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
overnight_root.mkdir(parents=True, exist_ok=True)
os.environ["THIMBLE_OUTPUT_ROOT"] = str(overnight_root)
os.environ.pop("THIMBLE_QUICK_TEST", None)

failures = []
for index, notebook in enumerate(resolved, start=1):
    print("\n" + "#" * 90)
    print(f"OVERNIGHT JOB {index}/2: {notebook.name}")
    print("#" * 90, flush=True)
    previous_cwd = Path.cwd()
    try:
        os.chdir(notebook.parent)
        ip.run_line_magic("run", f'"{notebook.name}"')
    except Exception as exc:
        failures.append((notebook.name, f"{type(exc).__name__}: {exc}"))
        print(traceback.format_exc(), flush=True)
    finally:
        os.chdir(previous_cwd)

print("\n" + "=" * 90)
print(f"OVERNIGHT SEQUENCE FINISHED. Results: {overnight_root.resolve()}")
if failures:
    print("Failures:")
    for name, message in failures:
        print(f"  {name}: {message}")
else:
    print("Both notebooks completed.")
